# 11 — Real Toronto sensory-dimension data: first exploration

First real step toward `sensory_weight()`: pulling real City of Toronto open data (not synthetic, not OSM tags) for the crowding dimension, starting with the **Traffic and Pedestrian Volumes** dataset. Construction data (Road Reconstruction, Road Resurfacing, Sidewalk Construction, Utility Cut Permits) follows the same two-step CKAN pattern once this one works.

See `research/sensory-accessibility-ontario.md` for the full research and the plan behind this module.

### Step 1 — get the dataset's package info

CKAN's `package_show` returns metadata and a list of `resources` — not the actual rows yet. Find the real dataset name (the `id` param) for pedestrian/traffic volumes on the Toronto Open Data catalogue, call `package_show`, and print each resource's `id`, `name`, and `format` so we know which resource to query next.

In [1]:
import requests
import sys
sys.path.append("..")

from src.graph.snapping import nearest_node_id

BASE_URL = "https://ckan0.cf.opendata.inter.prod-toronto.ca/api/3/action"

respound = requests.get(f"{BASE_URL}/package_show", params={"id": "traffic-volumes-at-intersections-for-all-modes"})
package = respound.json()["result"]

for resource in package["resources"]:
    print(resource["id"], "|", resource["name"], "|", resource["format"])


6afa3b1f-f6a5-4235-8bd6-7568411c19f4 | tmc_most_recent_summary_data | CSV
1364bffa-29a3-4c39-af8a-925d8ca7bf1f | tmc_summary_data | CSV
4402bb65-3936-49db-bf51-ec973381a5a3 | tmc_raw_data_1980_1989 | CSV
ce5ea7df-ee82-450e-aceb-2977ceeef66a | tmc_raw_data_1990_1999 | CSV
01aee77d-6c1a-4836-8fe7-b66d93b70ebb | tmc_raw_data_2000_2009 | CSV
ae400fe2-98ce-4e73-8a94-024b22e29a22 | tmc_raw_data_2010_2019 | CSV
262469c2-abfe-4756-9068-4ea5c7ba1af7 | tmc_raw_data_2020_2029 | CSV
96672ddd-387a-4686-bf2a-2e8d7e146a31 | tmc_raw_data_2010_2019.csv | CSV
72ae0814-0987-4487-9737-244457ac0796 | tmc_raw_data_2010_2019.xml | XML
601f48c3-8c99-4a3b-8cdf-3fe3b99a4db1 | tmc_raw_data_2010_2019.json | JSON
88ea1329-1da3-4992-bae6-a821d609d45d | tmc_raw_data_2020_2029.csv | CSV
35f15315-102f-40c7-9c8e-7cb0392bdf62 | tmc_raw_data_2020_2029.xml | XML
cd24d4b6-0a17-4e0b-82d8-71eede004cc7 | tmc_raw_data_2020_2029.json | JSON
d5999173-dd33-4156-a5c7-bbbf2d2763b8 | tmc_most_recent_summary_data.csv | CSV
858c4340-3

### Step 2 — pull real rows with `datastore_search`

Pick the resource id that actually holds tabular data (not a PDF/ZIP), and call `datastore_search` with a small `limit` first — we just want to see the real column names and a few sample rows before pulling everything.

In [2]:
resource_id = "6afa3b1f-f6a5-4235-8bd6-7568411c19f4"
resp = requests.get(f"{BASE_URL}/datastore_search", params={"id": resource_id, "limit": 5})
rows = resp.json()["result"]["records"]
for row in rows:
    print(row)

{'_id': 1, 'latest_count_id': 116519, 'latest_count_date': '2026-10-28', 'location_name': 'Bloor St W / The East Mall', 'longitude': -79.5567097191495, 'latitude': 43.6369296306976, 'centreline_type': '2', 'centreline_id': 13468751, 'px': 792, 'count_duration': '14', 'total_vehicle': 40948, 'total_bike': 204, 'total_heavy_pct': 0.026, 'total_pedestrian': 1261, 'am_peak_start': '2026-10-28T08:00:00', 'am_peak_vehicle': 3588, 'am_peak_bike': 14, 'am_peak_heavy_pct': 0.03, 'pm_peak_start': '2026-10-28T17:00:00', 'pm_peak_vehicle': 4009, 'pm_peak_bike': 21, 'pm_peak_heavy_pct': 0.016, 'n_appr_vehicle': 6198, 'n_appr_bike': 50, 'n_appr_heavy_pct': 0.028, 'e_appr_vehicle': 11658, 'e_appr_bike': 57, 'e_appr_heavy_pct': 0.028, 's_appr_vehicle': 9600, 's_appr_bike': 45, 's_appr_heavy_pct': 0.023, 'w_appr_vehicle': 13492, 'w_appr_bike': 52, 'w_appr_heavy_pct': 0.026}
{'_id': 2, 'latest_count_id': 117984, 'latest_count_date': '2026-10-27', 'location_name': 'Scarborough Golf Club Rd / Bankwell Ave

### Step 3 — what do we actually have?

Look at the real fields: is there a geographic location (lat/lon or an intersection name) we can match to OSM nodes/edges? Is there an actual volume/count number, and for what time period? Print the full field list and a few full rows, not just a glance — this decides whether the dataset is usable as-is or needs a different resource.

In [3]:
resp = requests.get(f"{BASE_URL}/datastore_search", params={"id": resource_id, "limit": 7000})
result = resp.json()["result"]
rows = result["records"]
print("rows fetched:", len(rows), "of", result["total"])

pedestrian_counts = [row["total_pedestrian"] for row in rows if row["total_pedestrian"] is not None]

import statistics
print("min:", min(pedestrian_counts))
print("max:", max(pedestrian_counts))
print("median:", statistics.median(pedestrian_counts))
quantiles = statistics.quantiles(pedestrian_counts, n=4)
print("quartiles (25/50/75):", quantiles)


rows fetched: 6431 of 6431
min: 0
max: 91808
median: 428
quartiles (25/50/75): [164.0, 428.0, 1202.0]


### Step 4 — bucket the real pedestrian counts

Use `pandas.cut()` with the real quartile boundaries (164, 1197.5) from the full 6,421-row dataset, instead of a hand-written if/elif chain.

In [4]:
import pandas as pd

crowding_bucket = pd.cut(
    pedestrian_counts,
    bins=[-1, 164, 1197.5, float("inf")],
    labels=["low", "medium", "high"]
)

print(pd.Series(crowding_bucket).value_counts())


medium    3205
low       1615
high      1611
Name: count, dtype: int64


### Step 5 — a reusable crowding-level function for an edge

Averages the raw pedestrian counts of an edge's two endpoint nodes first (reduces per-point noise), then buckets the average with the same real quartile thresholds.

In [5]:
from src.graph.sensory_weighting import crowding_level

# quick check with two real counts from the sample rows earlier
print(crowding_level(226, 3330))   # low + high endpoints -> averaged, then bucketed
print(crowding_level(50, 80))      # two quiet endpoints
print(crowding_level(2000, 5000))  # two busy endpoints


high
low
high


### Step 6 — snap a real Toronto intersection to the nearest OSM node

Generalized version of the toy example: works on any `osmnx`-style graph `G`, where every node has `"y"` (latitude) and `"x"` (longitude) attributes -- the same convention `accessibility_weight()` already relies on.

In [6]:
from src.graph.snapping import nearest_node_id

# Downtown Toronto's rough bounding box -- find a real intersection
# actually inside the graph's coverage area, not a random row.
downtown_rows = [
    row for row in rows
    if 43.63 <= row["latitude"] <= 43.68 and -79.41 <= row["longitude"] <= -79.36
]
print(len(downtown_rows), "rows inside the Downtown Toronto bounding box")

sample_row = downtown_rows[0]
target = (sample_row["latitude"], sample_row["longitude"])
print("target intersection:", sample_row["location_name"], target)


628 rows inside the Downtown Toronto bounding box
target intersection: Oak St / Regent St (43.6610461749136, -79.365018037479)


### Step 7 — load a real Toronto graph to snap onto

Same `osmnx` settings as Module 6 (Downtown Toronto, cached) -- reusing the already-downloaded area so this doesn't trigger a fresh, slow Overpass download.

In [7]:
import osmnx as ox

ox.settings.cache_folder = "cache"
ox.settings.useful_tags_way += ["wheelchair", "surface"]
ox.settings.useful_tags_node += ["kerb", "crossing", "tactile_paving"]
ox.settings.overpass_url = "https://overpass.kumi.systems/api/interpreter"

G = ox.graph_from_place("Downtown Toronto, Toronto, Ontario, Canada", network_type="walk")
print(G.number_of_nodes(), "nodes")

nearest_id = nearest_node_id(G, target, max_distance_m=200)
if nearest_id is None:
    print("no node within 200m -- this point is outside the graph's coverage")
else:
    print("nearest OSM node to", sample_row["location_name"], ":", nearest_id)
    print("node coords:", G.nodes[nearest_id]["y"], G.nodes[nearest_id]["x"])


14123 nodes
nearest OSM node to Oak St / Regent St : 9377844424
node coords: 43.6609742 -79.3650667


### Step 8 — crowding level for a real graph edge

Pick a real edge from `G`, snap each endpoint (`u`, `v`) to the nearest Toronto pedestrian-volume record with `nearest_row()`, then run both counts through `crowding_level()`. Requires both endpoints to have a real match within `max_distance_m` -- if either doesn't, we have no real crowding data for this edge and should say so, not guess.

In [8]:
from src.graph.sensory_weighting import edge_crowding

# try it on a real edge near our matched intersection
u, v, k = list(G.edges(keys=True))[0]
print("edge:", u, "->", v)
print("crowding level:", edge_crowding(G, u, v, downtown_rows))


edge: 20964445 -> 13433507992
crowding level: high


### Step 9 — a real low-sensory route on the Downtown Toronto graph

Compare a plain shortest path against `find_low_sensory_route()` between the same two real nodes, using the real pedestrian-volume data snapped onto the graph.

In [9]:
import networkx as nx
from src.routing.pathfinding import find_low_sensory_route

# pick two real, reasonably distant nodes in the Downtown Toronto graph
node_list = list(G.nodes)
source, target = node_list[0], node_list[len(node_list) // 2]

plain_path = nx.shortest_path(G, source, target, weight="length")
low_sensory_path, cost = find_low_sensory_route(G, source, target, downtown_rows)

print("plain shortest path length:", len(plain_path), "nodes")
print("low-sensory path length:", len(low_sensory_path), "nodes")
print("same path?", plain_path == low_sensory_path)
print("low-sensory route cost:", cost)


plain shortest path length: 111 nodes
low-sensory path length: 74 nodes
same path? False
low-sensory route cost: 16505.78912072524
